# Modeling
This notebook runs the modeling pipeline on the model‑ready EOM panel: walk‑forward splits, baselines, models, and backtesting.


## Setup


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from notebooks.params import OUT_DIR
from src.modeling.baseline import BaselineConfig, add_momentum_score
from src.modeling.backtest import BacktestConfig, backtest_from_scores, equal_weight_returns
from src.modeling.metrics import mean_monthly_spearman_ic, top_k_hit_rate
from src.modeling.models import ModelIO, build_ridge_pipeline, build_tree_model, fit_predict_oos_scores
from src.modeling.splits import WalkForwardConfig, generate_expanding_walk_forward_splits

STAGES_DIR = Path(OUT_DIR) / "stages"
MODEL_PATH = STAGES_DIR / "panel_model_ready.parquet"


## Load model‑ready panel


In [ ]:
df_model = pd.read_parquet(MODEL_PATH)
print("Rows:", len(df_model))
print("Date range:", df_model['date'].min(), "→", df_model['date'].max())
print("Tickers:", df_model['ticker'].nunique())


## Feature set
We exclude macro columns from training; they are used only for overlay/analysis.


In [ ]:
exclude_cols = {
    'date', 'ticker',
    'target_3m', 'target_1m',
    'fwd_ret_1m', 'fwd_ret_3m', 'fwd_ret_6m',
    'macro_regime', 'stress_index', 'slope_10y2y',
}
features = [c for c in df_model.columns if c not in exclude_cols]
print("Features:", features)


## Split strategy (expanding walk‑forward with embargo)


In [ ]:
split_cfg = WalkForwardConfig(
    train_years=5,
    test_months=12,
    embargo_months=3,
)

splits = list(generate_expanding_walk_forward_splits(df_model, split_cfg))
print("Folds:", len(splits))
print(splits[0][2] if splits else "No splits")


## Model selection


In [ ]:
io = ModelIO()

ridge = build_ridge_pipeline(alpha=1.0)
hgb = build_tree_model(max_depth=3, learning_rate=0.05, max_iter=300)


## Baseline scores


In [ ]:
baseline_cfg = BaselineConfig(momentum_col='mom12_pr', score_col='baseline_mom')
df_with_base = add_momentum_score(df_model, baseline_cfg)


## OOS scores (walk‑forward)


In [ ]:
oos_ridge, folds_ridge = fit_predict_oos_scores(
    splits=splits,
    features=features,
    model=ridge,
    io=io,
    score_col='score_ridge',
)

oos_hgb, folds_hgb = fit_predict_oos_scores(
    splits=splits,
    features=features,
    model=hgb,
    io=io,
    score_col='score_hgb',
)

# Merge baseline scores into OOS frames
base_scores = df_with_base[['date', 'ticker', 'baseline_mom']]

oos_ridge = oos_ridge.merge(base_scores, on=['date', 'ticker'], how='left')
oos_hgb = oos_hgb.merge(base_scores, on=['date', 'ticker'], how='left')


## Backtesting (Top‑K long‑only)


In [ ]:
n_tickers = df_model['ticker'].nunique()
if n_tickers >= 40:
    top_k = 5
elif n_tickers >= 25:
    top_k = 4
else:
    top_k = 3

bt_cfg = BacktestConfig(top_k=top_k)

score_cols = {
    'baseline_mom': 'baseline_mom',
    'ridge': 'score_ridge',
    'hgb': 'score_hgb',
}

summary_ridge, artifacts_ridge = backtest_from_scores(oos_ridge, score_cols, bt_cfg)
summary_hgb, artifacts_hgb = backtest_from_scores(oos_hgb, score_cols, bt_cfg)

# Equal‑weight benchmark
ew_ret = equal_weight_returns(oos_ridge, bt_cfg)
print("Equal‑weight mean monthly return:", ew_ret.mean())

summary = pd.concat([summary_ridge, summary_hgb]).drop_duplicates('model')
display(summary)


## Backtest report
Equity curve, drawdown and turnover for each model.


In [ ]:
def plot_report(artifacts, title_prefix):
    for name, art in artifacts.items():
        fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
        art['equity'].plot(ax=axes[0], title=f'{title_prefix} {name} - Equity')
        dd = art['equity'] / art['equity'].cummax() - 1.0
        dd.plot(ax=axes[1], title='Drawdown')
        if 'turnover' in art.columns:
            art['turnover'].plot(ax=axes[2], title='Turnover')
        plt.tight_layout()

plot_report(artifacts_ridge, 'Ridge fold‑OOS')
plot_report(artifacts_hgb, 'HGB fold‑OOS')


## IC and hit‑rate diagnostics


In [ ]:
def _diagnostics(df, score_col):
    ic = mean_monthly_spearman_ic(df, 'date', score_col, io.ret_col_for_ic)
    hit = top_k_hit_rate(df, 'date', score_col, io.ret_col_for_ic, top_k, ticker_col='ticker')
    return ic, hit

rows = []
for name, score in score_cols.items():
    ic, hit = _diagnostics(oos_ridge, score)
    rows.append({'model': name, 'ic_mean': ic, 'hit_rate': hit})

display(pd.DataFrame(rows))
